In [1]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install -q -U bitsandbytes peft trl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import os
import torch
import json
import ast
import matplotlib.pyplot as plt
from collections import defaultdict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig
from tqdm.notebook import tqdm

In [4]:
model_id = "NousResearch/Meta-Llama-3.1-8B-Instruct"

LORA_DIR  = "/kaggle/input/datasets/nguyentam306/data-llama3-1-2/Llama2-absa-final"  

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LORA_DIR)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)
base_model = prepare_model_for_kbit_training(base_model)

# Load LoRA weights đã train
model = PeftModel.from_pretrained(
    base_model,
    LORA_DIR,
    is_trainable=True,   
)

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

model.print_trainable_parameters()
model.config.pad_token_id = tokenizer.pad_token_id
print("✅ Load model đã train xong!")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

trainable params: 5,505,024 || all params: 8,035,766,272 || trainable%: 0.0685
✅ Load model đã train xong!


In [5]:
input_dir    = '/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data'
target_langs = ['en', 'vi', 'zh']
target_file  = 'clean_all.txt'
all_data     = []

print(f"Đang tìm kiếm file {target_file}...")

for root, dirs, files in os.walk(input_dir):
    folder_name = os.path.basename(root)
    if folder_name in target_langs:
        for file in files:
            if file == target_file:
                file_path = os.path.join(root, file)
                print(f"  Đọc: {folder_name}/{file}")
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        clean_line = line.strip()
                        if not clean_line or '####' not in clean_line:
                            continue
                        parts      = clean_line.split('####')
                        text_part  = parts[0].strip()
                        labels_part = parts[1].strip()
                        try:
                            true_labels = ast.literal_eval(labels_part)
                        except:
                            true_labels = []
                        all_data.append({
                            'language':    folder_name,
                            'file_name':   file,
                            'text':        text_part,
                            'true_labels': true_labels,
                            'domain':      os.path.basename(os.path.dirname(root))
                        })

df = pd.DataFrame(all_data)
print(f"\nTổng số mẫu đọc được: {len(df)}")

Đang tìm kiếm file clean_all.txt...
  Đọc: zh/clean_all.txt
  Đọc: vi/clean_all.txt
  Đọc: en/clean_all.txt
  Đọc: zh/clean_all.txt
  Đọc: vi/clean_all.txt
  Đọc: en/clean_all.txt
  Đọc: zh/clean_all.txt
  Đọc: vi/clean_all.txt
  Đọc: en/clean_all.txt
  Đọc: zh/clean_all.txt
  Đọc: vi/clean_all.txt
  Đọc: en/clean_all.txt
  Đọc: zh/clean_all.txt
  Đọc: vi/clean_all.txt
  Đọc: en/clean_all.txt
  Đọc: zh/clean_all.txt
  Đọc: vi/clean_all.txt
  Đọc: en/clean_all.txt
  Đọc: zh/clean_all.txt
  Đọc: vi/clean_all.txt
  Đọc: en/clean_all.txt

Tổng số mẫu đọc được: 38135


In [6]:
# ── CELL 5: Stratified sampling theo domain ──────────────────────
DOMAINS           = ['coursera', 'food', 'hotel', 'laptop', 'phone', 'restaurant', 'sight']
SAMPLE_PER_LANG   = 3000
SAMPLE_PER_DOMAIN = SAMPLE_PER_LANG // len(DOMAINS)

frames = []
for lang in df['language'].unique():
    sub   = df[df['language'] == lang]
    parts = []
    for domain in DOMAINS:
        domain_rows = sub[sub['domain'] == domain]
        n = min(SAMPLE_PER_DOMAIN, len(domain_rows))
        if n > 0:
            parts.append(domain_rows.sample(n=n, random_state=42))

    sampled   = pd.concat(parts)
    remaining = SAMPLE_PER_LANG - len(sampled)
    if remaining > 0:
        extra   = sub[~sub.index.isin(sampled.index)].sample(
            n=min(remaining, len(sub) - len(sampled)), random_state=42
        )
        sampled = pd.concat([sampled, extra])

    sampled = sampled.sample(frac=1, random_state=42).reset_index(drop=True)
    frames.append(sampled)
    print(f"{lang.upper()} ({len(sampled)} mẫu): {sampled['domain'].value_counts().to_dict()}")

df = pd.concat(frames).reset_index(drop=True)
print(f"\n✅ Tổng sau sampling: {len(df)} mẫu")

ZH (3000 mẫu): {'phone': 429, 'hotel': 429, 'restaurant': 429, 'sight': 429, 'coursera': 428, 'laptop': 428, 'food': 428}
VI (3000 mẫu): {'restaurant': 430, 'laptop': 430, 'phone': 428, 'coursera': 428, 'hotel': 428, 'sight': 428, 'food': 428}
EN (3000 mẫu): {'coursera': 430, 'sight': 429, 'restaurant': 429, 'hotel': 428, 'phone': 428, 'laptop': 428, 'food': 428}

✅ Tổng sau sampling: 9000 mẫu


In [7]:
# ── CELL 6: Normalize category + format labels ──────────────────
def normalize_category(cat):
    cat = str(cat).lower().strip()
    if '#' in cat:
        return cat
    parts = cat.rsplit(' ', 1)
    if len(parts) == 2:
        asp = parts[0].strip().replace(' ', '_')
        att = parts[1].strip().replace(' ', '_')
        return f"{asp}#{att}"
    return cat

def convert_to_dict_format(true_labels):
    out = []
    for t in true_labels:
        if len(t) == 3:
            out.append({
                'entity':    str(t[0]).strip(),
                'category':  normalize_category(t[1]),
                'sentiment': str(t[2]).lower().strip()
            })
    return out

df['formatted_true_labels'] = df['true_labels'].apply(convert_to_dict_format)

# Domain → category mapping
domain_to_cats = {}
for domain, group in df.groupby('domain'):
    cats = set()
    for labels in group['formatted_true_labels']:
        for lb in labels:
            cats.add(lb['category'])
    domain_to_cats[domain] = list(cats)

for domain, cats in domain_to_cats.items():
    print(f"  {domain}: {len(cats)} categories")

my_categories = list({
    lb['category']
    for labels in df['formatted_true_labels']
    for lb in labels
})
print(f"\n✅ Tổng {len(my_categories)} categories")

  coursera: 28 categories
  food: 10 categories
  hotel: 34 categories
  laptop: 75 categories
  phone: 83 categories
  restaurant: 13 categories
  sight: 5 categories

✅ Tổng 241 categories


In [8]:

def group_categories(category_list):
    groups = defaultdict(list)
    for cat in category_list:
        if '#' not in cat:
            continue
        asp, att = cat.split('#', 1)
        groups[asp].append(att)
    return "\n".join(
        f"  {asp}: [{', '.join(sorted(atts))}]"
        for asp, atts in sorted(groups.items())
    )
def build_user_prompt(text, category_list, lang="en"):
    cat_block = group_categories(category_list)
    lang_rule = {
        "zh": "Extract entity in Chinese. Do NOT translate.",
        "vi": "Extract entity in Vietnamese. Do NOT translate.",
        "en": ""
    }.get(lang, "")

    return f"""Extract ABSA tuples from the review. {lang_rule}
Categories: {cat_block}
Return ONLY a JSON array. entity=NULL if not explicitly named.

Text: "{text}"
Output:"""

In [9]:
# ── CELL 8 ──────────────────────────────────────────────────────
TRAIN_PER_LANG = 334
VAL_PER_LANG   = 167
TEST_PER_LANG  = 667

# Lấy train
# 500 mẫu cũ (giữ nguyên random_state=42)
old_train = (
    df.groupby('language', group_keys=False)
      .apply(lambda x: x.sample(n=167, random_state=42))
      .reset_index(drop=True)
)
# 500 mẫu mới
remaining = df[~df.index.isin(old_train.index)]
new_train = (
    remaining.groupby('language', group_keys=False)
             .apply(lambda x: x.sample(n=167, random_state=42))
             .reset_index(drop=True)
)
train_df = pd.concat([old_train, new_train]).sample(frac=1, random_state=42).reset_index(drop=True)

# Lấy val từ phần còn lại
remaining = df[~df.index.isin(train_df.index)]
val_df = (
    remaining.groupby('language', group_keys=False)
             .apply(lambda x: x.sample(n=VAL_PER_LANG, random_state=42))  # ← bỏ include_groups
             .reset_index(drop=True)
)

# Lấy test từ phần còn lại sau val
remaining2 = remaining[~remaining.index.isin(val_df.index)]
test_df = (
    remaining2.groupby('language', group_keys=False)
              .apply(lambda x: x.sample(n=TEST_PER_LANG, random_state=42))  # ← bỏ include_groups
              .reset_index(drop=True)
)

print(f"Train: {len(train_df)} — {train_df['language'].value_counts().to_dict()}")
print(f"Val:   {len(val_df)}   — {val_df['language'].value_counts().to_dict()}")
print(f"Test:  {len(test_df)}  — {test_df['language'].value_counts().to_dict()}")

Train: 1002 — {'en': 334, 'vi': 334, 'zh': 334}
Val:   501   — {'en': 167, 'vi': 167, 'zh': 167}
Test:  2001  — {'en': 667, 'vi': 667, 'zh': 667}


/tmp/ipykernel_24/3883133069.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=167, random_state=42))
/tmp/ipykernel_24/3883133069.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=167, random_state=42))
/tmp/ipykernel_24/3883133069.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future v

In [10]:
# ── CELL 10: Tạo training samples ───────────────────────────────
def make_training_sample(row):
    text = row['text']
    lang = row['language']
    lang_rule = {
        "zh": "Extract entity in Chinese. Do NOT translate.",
        "vi": "Extract entity in Vietnamese. Do NOT translate.",
        "en": ""
    }.get(lang, "")
    user_content = f"""Extract ABSA tuples from the review. {lang_rule}
Return ONLY a JSON array: [{{"entity":"...","category":"...","sentiment":"..."}}]
entity=NULL if not explicitly named. sentiment: positive|negative|neutral.
Text: "{text}"
Output:"""
    messages    = [{"role": "user", "content": user_content}]
    prompt_part = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    output_json = json.dumps(row['formatted_true_labels'], ensure_ascii=False)
    return prompt_part + output_json + tokenizer.eos_token

print("Tạo training samples...")
train_df['training_text'] = train_df.apply(make_training_sample, axis=1)
val_df['training_text']   = val_df.apply(make_training_sample,   axis=1)   # ← thêm
test_df['training_text']  = test_df.apply(make_training_sample,  axis=1)

# Lọc mẫu quá dài tránh OOM
MAX_LEN = 1536
for _df in [train_df, val_df, test_df]:
    _df['token_len'] = _df['training_text'].apply(
        lambda x: len(tokenizer.encode(x, add_special_tokens=False))
    )

train_df = train_df[train_df['token_len'] <= MAX_LEN].reset_index(drop=True)
val_df   = val_df[val_df['token_len']     <= MAX_LEN].reset_index(drop=True)
test_df  = test_df[test_df['token_len']   <= MAX_LEN].reset_index(drop=True)

print(f"Sau lọc — Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Token length — mean={train_df['token_len'].mean():.0f}, max={train_df['token_len'].max()}")

Tạo training samples...
Sau lọc — Train: 1002, Val: 501, Test: 2001
Token length — mean=130, max=405


In [11]:
from datasets import Dataset as HFDataset

train_dataset = HFDataset.from_dict({"text": train_df['training_text'].tolist()})
val_dataset   = HFDataset.from_dict({"text": val_df['training_text'].tolist()})   # ← thêm
test_dataset  = HFDataset.from_dict({"text": test_df['training_text'].tolist()})

print(f"✅ train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

✅ train=1002, val=501, test=2001


In [12]:
print(f"Max token length trong train set: {train_df['token_len'].max()}")
print(f"Mean token length: {train_df['token_len'].mean():.0f}")

Max token length trong train set: 405
Mean token length: 130


In [13]:
from transformers import DataCollatorForLanguageModeling

# ── CELL 13: Training ────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/gemma2-absa-lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",

    # Epochs & batch
    num_train_epochs=2,             # tăng lên 7 nếu loss vẫn giảm
    per_device_train_batch_size=2,  # T4/P100: giữ 2
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # effective batch = 2×8 = 16
    max_grad_norm=0.3,
    
    # Optimizer
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    optim="paged_adamw_8bit",

    # Eval & save
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Tốc độ
    fp16=False,    
    bf16=False, 
    dataloader_num_workers=2,
    group_by_length=True,
    # max_seq_length=MAX_LEN,

    # Log
    logging_steps=1,
    disable_tqdm=False,  
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(   # ← thêm
        tokenizer=tokenizer,
        mlm=False,   # causal LM không dùng masked LM
    ),
)

print("🚀 Bắt đầu fine-tune...")
train_result = trainer.train()

print("\n=== KẾT QUẢ TRAINING ===")
print(f"Train loss: {train_result.training_loss:.4f}")
metrics = trainer.evaluate()
print(f"Eval loss:  {metrics['eval_loss']:.4f}")

Adding EOS to train dataset:   0%|          | 0/1002 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1002 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/501 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/501 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


🚀 Bắt đầu fine-tune...


Epoch,Training Loss,Validation Loss
1,0.257805,0.528478
2,0.188788,0.515406



=== KẾT QUẢ TRAINING ===
Train loss: 0.5176


Eval loss:  0.5154


In [14]:
# ── CELL 14: Lưu model ──────────────────────────────────────────
SAVE_DIR = "/kaggle/working/Llama-3.1-absa-final"

trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Đã lưu LoRA adapter tại: {SAVE_DIR}")

for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")


✅ Đã lưu LoRA adapter tại: /kaggle/working/Llama-3.1-absa-final
  adapter_config.json: 0.0 MB
  tokenizer.json: 16.4 MB
  chat_template.jinja: 0.0 MB
  README.md: 0.0 MB
  adapter_model.safetensors: 21.0 MB
  tokenizer_config.json: 0.0 MB


In [15]:
# ── CELL 16: Inference batch + tính F1 ──────────────────────────
model.eval()
tokenizer.padding_side = "left"

def batch_inference(test_df, batch_size=8):
    # Build tất cả prompt
    prompts = []
    for _, row in test_df.iterrows():
        lang_rule = {
            "zh": "Extract entity in Chinese. Do NOT translate.",
            "vi": "Extract entity in Vietnamese. Do NOT translate.",
            "en": ""
        }.get(row['language'], "")

        user_content = f"""Extract ABSA tuples from the review. {lang_rule}
Return ONLY a JSON array: [{{"entity":"...","category":"...","sentiment":"..."}}]
entity=NULL if not explicitly named. sentiment: positive|negative|neutral.
Text: "{row['text']}"
Output:"""
        messages = [{"role": "user", "content": user_content}]
        prompts.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        ))

    # Sort theo độ dài giảm padding
    lengths = [len(tokenizer.encode(p)) for p in prompts]
    sorted_idx = sorted(range(len(prompts)), key=lambda i: lengths[i])
    sorted_prompts = [prompts[i] for i in sorted_idx]

    # Batch generate
    all_responses = [None] * len(prompts)
    for i in tqdm(range(0, len(sorted_prompts), batch_size), desc="Inference"):
        batch = sorted_prompts[i : i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt",
            padding=True, truncation=True, max_length=512
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        new_tokens = [
            out[inputs['input_ids'].shape[1]:]
            for out in output
        ]
        responses = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        for j, resp in enumerate(responses):
            all_responses[sorted_idx[i + j]] = resp

        del inputs, output, new_tokens
        torch.cuda.empty_cache()

    return all_responses

# Chạy inference
raw_responses = batch_inference(test_df, batch_size=8)

# Tính F1
metrics_by_lang = {lang: {'tp': 0, 'fp': 0, 'fn': 0} for lang in ['zh', 'vi', 'en']}
comparison_data = []

for i, (_, row) in enumerate(test_df.iterrows()):
    try:
        pred = json.loads(raw_responses[i].replace("```json","").replace("```","").strip())
        if not isinstance(pred, list): pred = []
    except: pred = []

    lang = row['language']
    true_set = set(
        (str(t.get('entity','')).lower().strip(),
         str(t.get('category','')).lower().strip(),
         str(t.get('sentiment','')).lower().strip())
        for t in row['formatted_true_labels']
    )
    pred_set = set(
        (str(p.get('entity','')).lower().strip(),
         str(p.get('category','')).lower().strip(),
         str(p.get('sentiment','')).lower().strip())
        for p in pred if isinstance(p, dict)
    )
    metrics_by_lang[lang]['tp'] += len(true_set & pred_set)
    metrics_by_lang[lang]['fp'] += len(pred_set - true_set)
    metrics_by_lang[lang]['fn'] += len(true_set - pred_set)
    comparison_data.append({
        'Language':         lang,
        'Text':             row['text'],
        'True_Labels':      json.dumps(row['formatted_true_labels'], ensure_ascii=False),
        'Predicted_Labels': json.dumps(pred, ensure_ascii=False)
    })

# In kết quả
print("\n--- KẾT QUẢ SAU FINE-TUNE ---")
for lang, m in metrics_by_lang.items():
    pr = m['tp']/(m['tp']+m['fp']) if m['tp']+m['fp'] else 0
    rc = m['tp']/(m['tp']+m['fn']) if m['tp']+m['fn'] else 0
    f1 = 2*pr*rc/(pr+rc) if pr+rc else 0
    print(f"{lang.upper()} | Precision={pr:.3f} Recall={rc:.3f} F1={f1:.3f}")

# Lưu CSV
pd.DataFrame(comparison_data).to_csv(
    '/kaggle/working/absa_finetuned_results.csv',
    index=False, encoding='utf-8-sig'
)
print("✅ Lưu tại: /kaggle/working/absa_finetuned_results.csv")

Inference:   0%|          | 0/251 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- KẾT QUẢ SAU FINE-TUNE ---
ZH | Precision=0.386 Recall=0.358 F1=0.372
VI | Precision=0.436 Recall=0.400 F1=0.417
EN | Precision=0.455 Recall=0.438 F1=0.446
✅ Lưu tại: /kaggle/working/absa_finetuned_results.csv
